Training a Llama-3-8B model using Unsloth and TRL for the Enterprise Email Triage environment.

## Overview
- **Model**: Llama-3-8B-Instruct with 4-bit quantization
- **Environment**: EnterpriseEmailEnv with LLM tool calling
- **Framework**: TRL (Transformers Reinforcement Learning)
- **Optimization**: LoRA adapters with r=16
- **Target**: Learn optimal email triage decisions

In [ ]:
# ==========================================
# 1. THE STABLE ENVIRONMENT INSTALL
# ==========================================
import os
import subprocess

# Install dependencies properly
subprocess.run(['pip', 'install', 'torch>=2.4.0,<2.11.0', 'torchvision', 'torchaudio'])
subprocess.run(['pip', 'install', 'unsloth', 'unsloth_zoo'])
subprocess.run(['pip', 'install', 'trl==0.22.0', 'peft', 'accelerate', 'bitsandbytes', 'openenv-core', 'datasets'])

import json
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import PPOConfig, PPOTrainer
from peft import LoraConfig

# Download environment files
!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/main/env.py || \
!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/master/env.py

from env import EnterpriseEmailEnv

# ==========================================
# 2. LOAD THE FAST 3B MODEL
# ==========================================
print("Loading Unsloth 3B Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)

# Apply LoRA adapters correctly
lora_config = LoraConfig(
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    task_type="CAUSAL_LM",
)

model = FastLanguageModel.get_peft_model(model, lora_config)

# ==========================================
# 3. GENERATE ROLLOUTS & PREPARE DATASET
# ==========================================
print("Initializing OpenEnv...")
env = EnterpriseEmailEnv()

def collect_rollouts(env, model, tokenizer, num_episodes=30):
    """Collect rollouts from environment"""
    rollout_data = []
    
    for episode in range(num_episodes):
        obs = env.reset()
        
        # Format observation for model
        prompt = f"""You are an Email Triage Agent. Analyze this email and respond with JSON tool call.

Email: {json.dumps(obs, indent=2)}

Available tools: auto_reply, route_to_human, ask_for_clarification
Response format: {{"tool": "tool_name", "arguments": {{"key": "value"}}}

Your response:"""

        # Generate action
        inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=80,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        # Parse action
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        try:
            action = json.loads(generated_text.split("Your response:")[-1].strip())
            if "tool" not in action:
                action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs.get('current_email', {}).get('email_id', 'unknown')}}
        except:
            action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs.get('current_email', {}).get('email_id', 'unknown')}}
        
        # Take action in environment
        next_obs, reward, done, info = env.step(action)
        
        rollout_data.append({
            "prompt": prompt,
            "action": action,
            "reward": reward,
            "observation": next_obs
        })
        
        if done:
            break
    
    return rollout_data

print("Collecting rollouts (playing game)...")
rollout_data = collect_rollouts(env, model, tokenizer, num_episodes=30)
print(f"Collected {len(rollout_data)} transitions")

def prepare_dataset(rollout_data):
    dataset_dict = {
        "prompt": [item["prompt"] for item in rollout_data],
        "chosen": [json.dumps(item["action"]) for item in rollout_data],
        "reward": [item["reward"] for item in rollout_data],
    }
    dataset = Dataset.from_dict(dataset_dict)
    
    def tokenize_data(examples):
        full_texts = [p + " " + c for p, c in zip(examples["prompt"], examples["chosen"])]
        return tokenizer(full_texts, padding="max_length", truncation=True, max_length=512)
        
    tokenized_dataset = dataset.map(
        tokenize_data, 
        batched=True,
        remove_columns=["prompt", "chosen"] 
    )
    tokenized_dataset.set_format(type="torch")
    return tokenized_dataset

train_dataset = prepare_dataset(rollout_data)

In [ ]:
# ==========================================
# 4. CONFIGURE PPO FOR A10G HARDWARE
# ==========================================
ppo_config = PPOConfig(
    batch_size=8,                  # Increased to 8 since you have 24GB VRAM
    mini_batch_size=4,             
    gradient_accumulation_steps=2,
    learning_rate=1.4e-5,
    max_grad_norm=1.0,
    optimize_cuda_cache=True,
    init_kl_coef=0.2,
    adap_kl_ctrl=True,
    ppo_epochs=4,
    cliprange=0.2,
    vf_coef=0.5,
    scale_reward=False,
    # Removed TensorBoard logging to prevent Accelerate crashes
)

# ==========================================
# 5. INITIALIZE TRAINER (THE HYBRID FIX)
# ==========================================
print("Initializing PPO Trainer...")
ppo_trainer = PPOTrainer(
    config=ppo_config,              # Fixed: config not args
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    dataset=train_dataset,  
)

# ==========================================
# 6. EXECUTE TRAINING & SAVE DELIVERABLES
# ==========================================
print("Starting PPO training (This will take a few minutes)...")
ppo_trainer.train()

print("Training completed! Saving LoRA adapters...")
# Save trained weights to a folder so you can submit them
model.save_pretrained("email-triage-lora-final")
tokenizer.save_pretrained("email-triage-lora-final")

print("SUCCESS: Check your file browser for 'email-triage-lora-final' folder!")

In [ ]:
# Cell 3: Environment Setup with Strong Reward System
import sys
import os
from env import EnterpriseEmailEnv
import numpy as np
from typing import Dict, Any
from reward_system import reward_system, RewardBreakdown

# Initialize environment
env = EnterpriseEmailEnv()
observation = env.reset()
print(f"Environment initialized with {len(env.emails)} emails")

def augment_prompt(original_prompt: str, augmentation_level: int = 3) -> str:
    """
    Aggressively augment prompts to prevent memorization
    """
    augmentations = [
        # Add noise
        lambda p: p.replace("urgent", "critical").replace("important", "priority"),
        lambda p: p.replace("please", "kindly").replace("help", "assist"),
        lambda p: p.replace("email", "message").replace("request", "inquiry"),
        # Change order
        lambda p: " ".join(random.sample(p.split(), len(p.split()))),
        # Add filler
        lambda p: p + " FYI this requires attention.",
        lambda p: "Note: " + p,
        # Change perspective
        lambda p: p.replace("I need", "We require").replace("my", "our"),
    ]
    
    augmented = original_prompt
    for _ in range(augmentation_level):
        aug = random.choice(augmentations)
        augmented = aug(augmented)
    
    return augmented

def format_observation_to_prompt(observation: Dict[str, Any], augment: bool = False) -> str:
    """
    Convert environment observation to prompt with optional augmentation
    """
    current_email = observation['current_email']
    
    # Base system prompt
    system_prompt = """You are an Enterprise Email Triage Agent. Analyze emails and respond with JSON tool calls.

Available tools: auto_reply, route_to_human, ask_for_clarification

Response format: {"tool": "tool_name", "arguments": {"key": "value"}}

Rules:
- route_to_human requires "department" argument
- auto_reply requires "message" argument  
- All actions require "email_id" argument
- Be decisive and accurate"""
    
    # Create user prompt with augmentation
    email_content = current_email.get('content', current_email.get('body', ''))
    if augment:
        email_content = augment_prompt(email_content)
    
    user_prompt = f"""Email to triage:
From: {current_email.get('sender', 'Unknown')}
Subject: {current_email.get('subject', 'No Subject')}
Content: {email_content}
Email ID: {current_email.get('email_id', 'unknown')}

Analyze and take appropriate action."""
    
    # Combine for model input
    full_prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n{user_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    
    return full_prompt

# Download reward system if needed
!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/main/reward_system.py || \
!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/master/reward_system.py

In [ ]:
# Cell 4: Advanced Rollout Collection with Temperature Variation
from trl import PPOTrainer, PPOConfig
from transformers import TrainingArguments
from datasets import Dataset
import torch.nn.functional as F
from tqdm import tqdm
import time
import random

# Training configuration optimized for 3B model
training_config = PPOConfig(
    batch_size=8,  # Increased for more samples
    mini_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,  # Higher learning rate for 3B
    max_grad_norm=1.0,
    report_to="none",
    optimize_device_cache=True,  # Important for 3B
)

def generate_trajectory_with_temperature(env, model, tokenizer, base_observation, temperature):
    """
    Generate a complete trajectory with given temperature
    """
    trajectory = []
    obs = base_observation.copy()
    
    # Reset environment for new trajectory
    env.reset()
    
    for step in range(5):  # 5 steps per trajectory
        # Format with augmentation
        augment = random.random() < 0.7  # 70% augmentation
        prompt = format_observation_to_prompt(obs, augment=augment)
        
        # Generate with specific temperature
        inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=80,
                temperature=temperature,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        # Parse action
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        json_match = re.search(r'\{.*\}', generated_text)
        
        if json_match:
            try:
                action = json.loads(json_match.group())
                if "tool" not in action or "arguments" not in action:
                    action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}
            except:
                action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}
        else:
            action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}
        
        # Calculate strong reward
        reward_breakdown = reward_system.calculate_reward(obs, action)
        reward = reward_breakdown.total_reward
        
        # Store trajectory step
        trajectory.append({
            "prompt": prompt,
            "action": action,
            "reward": reward,
            "temperature": temperature,
            "breakdown": reward_breakdown
        })
        
        # Next observation
        next_obs, _, done, _ = env.step(action)
        obs = next_obs
        
        if done or step >= 4:
            break
    
    return trajectory

def collect_diverse_rollouts(env, model, tokenizer, num_base_samples=100):
    """
    Collect diverse rollouts from 100 base samples with temperature variation
    """
    all_trajectories = []
    temperatures = [0.1, 0.3, 0.5, 0.7, 0.9, 1.1]  # Temperature range
    
    print(f"Generating trajectories from {num_base_samples} base samples...")
    
    for i in tqdm(range(num_base_samples), desc="Collecting diverse rollouts"):
        # Get base observation
        obs = env.reset()
        
        # Generate multiple trajectories per sample with different temperatures
        for temp in temperatures:
            trajectory = generate_trajectory_with_temperature(env, model, tokenizer, obs, temp)
            all_trajectories.extend(trajectory)
        
        # Shuffle every 10 samples to prevent order effects
        if (i + 1) % 10 == 0:
            random.shuffle(all_trajectories)
            print(f"Shuffled after {i+1} samples, total trajectories: {len(all_trajectories)}")
    
    # Final shuffle
    random.shuffle(all_trajectories)
    print(f"Collected {len(all_trajectories)} trajectory steps from {num_base_samples} base samples")
    
    return all_trajectories

In [ ]:
# Cell 5: Supervised Fine-Tuning → PPO Strategy
import torch
from torch.optim import AdamW
from datasets import Dataset
import numpy as np

def prepare_sft_dataset(trajectories):
    """
    Prepare dataset for supervised fine-tuning first
    """
    sft_data = []
    
    for traj in trajectories:
        # Only use high-quality examples for SFT
        if traj['reward'] > 0.5:  # Positive examples only
            sft_data.append({
                "instruction": traj['prompt'].split("Email to triage:")[0],
                "input": "Email to triage: " + traj['prompt'].split("Email to triage:")[1],
                "output": json.dumps(traj['action'])
            })
    
    return Dataset.from_list(sft_data)

def prepare_ppo_dataset(trajectories):
    """
    Prepare dataset for PPO training
    """
    ppo_data = []
    
    for traj in trajectories:
        ppo_data.append({
            "query": traj['prompt'],
            "response": json.dumps(traj['action']),
            "reward": traj['reward'],
            "temperature": traj['temperature']
        })
    
    return Dataset.from_list(ppo_data)

# Step 1: Collect diverse trajectories
print("Step 1: Collecting diverse trajectories...")
trajectories = collect_diverse_rollouts(env, model, tokenizer, num_base_samples=100)

print(f"Collected {len(trajectories)} trajectory steps")
print(f"Average reward: {np.mean([t['reward'] for t in trajectories]):.3f}")
print(f"Reward distribution: {np.histogram([t['reward'] for t in trajectories], bins=5)[0]}")

# Step 2: Supervised Fine-Tuning
print("\nStep 2: Supervised Fine-Tuning...")
sft_dataset = prepare_sft_dataset(trajectories)
print(f"SFT dataset size: {len(sft_dataset)}")

# Simple SFT training loop
optimizer = AdamW(model.parameters(), lr=2e-5)
model.train()

sft_epochs = 3
for epoch in range(sft_epochs):
    total_loss = 0
    for i, batch in enumerate(sft_dataset):
        # Prepare inputs
        full_text = f"{batch['instruction']}\n\n{batch['input']}\n\n{batch['output']}"
        inputs = tokenizer(full_text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
        
        # Forward pass
        outputs = model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss
        
        # Backward pass
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
        
        if i % 20 == 0:
            print(f"SFT Epoch {epoch+1}, Step {i}, Loss: {loss.item():.4f}")
    
    avg_loss = total_loss / len(sft_dataset)
    print(f"SFT Epoch {epoch+1} completed, Avg Loss: {avg_loss:.4f}")

print("Supervised Fine-Tuning completed!")

# Step 3: PPO Training
print("\nStep 3: PPO Training...")
ppo_dataset = prepare_ppo_dataset(trajectories)
print(f"PPO dataset size: {len(ppo_dataset)}")

# Initialize PPO trainer
ppo_trainer = PPOTrainer(
    config=training_config,
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    dataset=ppo_dataset,
)

# PPO Training
try:
    print("Starting PPO training...")
    ppo_trainer.train()
    print("PPO training completed successfully!")
except Exception as e:
    print(f"PPO training failed: {e}")
    print("Falling back to simple reward-based fine-tuning...")
    
    # Fallback: Simple reward-weighted training
    for epoch in range(2):
        total_loss = 0
        for i, batch in enumerate(ppo_dataset):
            # Weight loss by reward
            weight = max(0.1, abs(batch['reward']))
            
            full_text = f"{batch['query']}\n{batch['response']}"
            inputs = tokenizer(full_text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
            
            outputs = model(**inputs, labels=inputs["input_ids"])
            weighted_loss = outputs.loss * weight
            
            weighted_loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            total_loss += weighted_loss.item()
        
        avg_loss = total_loss / len(ppo_dataset)
        print(f"Fallback Epoch {epoch+1}, Avg Weighted Loss: {avg_loss:.4f}")

print("Training completed!")

In [ ]:
# Cell 6: Save the Adapter
# Create output directory
!mkdir -p ./trained_email_agent

# Save LoRA adapter
adapter_path = "./trained_email_agent/lora_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"LoRA adapter saved to {adapter_path}")
print("Training complete! Model is ready for deployment.")

In [ ]:
# ==========================================
# CELL 7: GENERATE SUBMISSION GRAPHS
# ==========================================
import matplotlib.pyplot as plt
import pandas as pd

print("Extracting training logs...")
# Extract the log history from the trainer
logs = ppo_trainer.state.log_history
df = pd.DataFrame(logs)

# Drop rows that don't have step data
df = df.dropna(subset=['step'])

# --- 1. PLOT REWARD CURVE ---
plt.figure(figsize=(10, 5))
# TRL sometimes names the reward metric differently based on config, checking common names:
reward_col = next((col for col in df.columns if 'reward' in col.lower()), None)

if reward_col:
    plt.plot(df['step'], df[reward_col], label='Model Reward', color='#2ecc71', linewidth=2)
    plt.title('Agent Reward Over Time (Learning to Route)', fontsize=14)
    plt.xlabel('Training Steps', fontsize=12)
    plt.ylabel('Reward Score', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.savefig('reward_curve.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved reward_curve.png (plotted using {reward_col})")
else:
    print("⚠️ Could not find a reward column in the logs. Check df.columns to see available metrics.")

# --- 2. PLOT LOSS CURVE ---
plt.figure(figsize=(10, 5))
if 'loss' in df.columns:
    plt.plot(df['step'], df['loss'], label='Training Loss', color='#e74c3c', linewidth=2)
    plt.title('Model Loss Over Time', fontsize=14)
    plt.xlabel('Training Steps', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.savefig('loss_curve.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Saved loss_curve.png")
else:
    print("⚠️ Could not find 'loss' in the logs.")

print("🎉 All submission graphs are ready! You can download them from the folder icon on the left.")

In [ ]:
# ==========================================
# CELL 8: SAVE THE LORA ADAPTERS
# ==========================================

# 1. Save a local backup to Colab's hard drive (Instant & Safe)
print("Saving local backup...")
model.save_pretrained("email_triage_lora_backup")
tokenizer.save_pretrained("email_triage_lora_backup")
print("Local backup saved!")

# 2. Push to Hugging Face (Uncomment these and add your details)
# print("Pushing to Hugging Face...")
# HF_TOKEN = "your_hf_write_token_here" # Get this from huggingface.co/settings/tokens
# HF_REPO_NAME = "your-username/email-triage-agent"
# 
# model.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
# print("Successfully uploaded to Hugging Face!")

## Training Summary

This notebook trains a Llama-3-8B model to perform email triage using reinforcement learning:

1. **Model Setup**: 4-bit quantized Llama-3-8B with LoRA (r=16)
2. **Environment Integration**: EnterpriseEmailEnv with JSON tool-calling format
3. **Data Collection**: Rollouts with reward tracking
4. **PPO Training**: Using TRL for policy optimization
5. **Adapter Saving**: LoRA weights for deployment

The trained model learns to:
- Route VIP emergencies to Emergency Support
- Auto-reply to password resets appropriately
- Detect phishing attempts and route to Security
- Handle HR issues with empathy (route to HR)
- Use clarification for ambiguous requests

## Usage

After training, load the model with:
```python
from unsloth import FastLanguageModel
from peft import PeftModel

base_model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    load_in_4bit=True
)
model = PeftModel.from_pretrained(base_model, "./trained_email_agent/lora_adapter")
```